# LSTM Next Word Predictor

A simple PyTorch LSTM model that learns to predict the **next word** given some input text.

The steps we follow are:
1. Install & import libraries
2. Load the text data
3. Preprocess the text (tokenize, build vocabulary)
4. Build training sequences
5. Pad the sequences
6. Create the PyTorch Dataset & DataLoader
7. Define the LSTM model
8. Train the model
9. Check accuracy
10. Generate predictions
11. Save the model

## 1. Install & Import Libraries

In [ ]:
!pip install nltk

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

## 2. Load the Text Data

This is the raw document our model will learn word patterns from. We use a longer FAQ document (about a Full Stack Web Development Bootcamp) so the model has more text to learn from.

In [3]:
document = """About the Bootcamp
What is the Full Stack Web Development Bootcamp?
The Full Stack Web Development Bootcamp is an online program that teaches you how to build complete websites and web applications from scratch.
Who is this bootcamp for?
This bootcamp is for beginners as well as working professionals who want to switch their career into web development.
What is the total duration of the bootcamp?
The total duration of the bootcamp is 6 months, with live classes held three times every week.
What is the course fee for the bootcamp?
The course fee is Rs 999 per month, and you can pay it as a monthly subscription.
What is the total cost if I pay for all 6 months?
The total cost becomes 999 times 6 which is Rs 5994 approximately.
What topics are covered in the bootcamp?
We cover the following modules in detail.
HTML and CSS Basics
JavaScript Fundamentals
React for Frontend Development
Node js and Express for Backend Development
Databases using MongoDB and SQL
Git and GitHub for Version Control
Deployment and Hosting
Building Real World Projects
Will I learn about databases in this bootcamp?
Yes, you will learn both SQL and NoSQL databases including MongoDB and PostgreSQL.
Will I build real projects during the bootcamp?
Yes, you will build at least five real world projects including a portfolio website, a blog application, an online store, a chat application and a final capstone project.
Do I need any prior coding experience to join?
No, you do not need any prior coding experience, we start from the very basics.
What if I miss a live class?
All our live classes are recorded and uploaded to your dashboard, so you can watch them anytime.
How can I ask doubts during the bootcamp?
You can ask your doubts in the live class chat or you can post them in our community discussion forum.
Will there be doubt clearing sessions?
Yes, we conduct one on one doubt clearing sessions every weekend for all enrolled students.
What is the class schedule for the bootcamp?
Classes are held every Monday, Wednesday and Friday from 7 PM to 9 PM.
What language is used to teach the bootcamp?
The bootcamp is taught in Hinglish so that both English and Hindi speaking students can understand easily.
Can I join the bootcamp if I am from a non technical background?
Yes, absolutely, many of our successful students come from non technical backgrounds.
Can I join the bootcamp after it has already started?
Yes, you can join anytime and you will get access to all the past recorded lectures.
Will I get access to past lectures if I join late?
Yes, once you complete the payment you will get access to all the past lectures on your dashboard.
Where do I need to submit my project assignments?
You need to submit all your assignments on our learning portal under the assignments section.
Will you provide solutions for the assignments?
Yes, we provide detailed solutions for every assignment so you can self evaluate your work.
Will there be any case studies in the bootcamp?
Yes, we cover several case studies based on real companies and real world problems.
Where can I contact the support team?
You can send an email to support at fullstackacademy dot com for any queries.
Payment and Registration Questions
Where do I need to make my monthly payments?
You need to make all your monthly payments through our official website only.
Can I pay the entire fee amount in one single payment?
Yes, you can choose to pay the entire fee amount of Rs 5994 in a single payment if you prefer.
What is the validity period of the monthly subscription?
The validity period is 30 days starting from the day you make the payment.
What happens if I do not like the bootcamp after paying?
We offer a 7 day refund policy starting from the day you make your first payment.
I am living outside India, how can I make the payment?
You can contact our support team by email and we will help you complete the payment process.
Post Registration Questions
Till when can I access the paid videos on the website?
You can access the paid videos as long as your monthly subscription remains active.
What happens after my subscription expires?
After your subscription expires you will need to renew it to continue watching new sessions, but once you complete the full course you get lifetime access.
Why is lifetime access only given after completing the full course?
This helps us keep the monthly course fee low for everyone who is still learning.
Can I still ask questions from previous weeks if I joined late?
Yes, you can ask questions from any previous week in the doubt clearing session.
Certificate and Placement Questions
What is required to receive the completion certificate?
There are two requirements, you must pay the entire course fee and you must complete all the project assignments.
Does the bootcamp guarantee placement after completion?
No, we provide placement assistance but we do not guarantee any job or interview calls.
What does placement assistance actually include?
Placement assistance includes resume building sessions, mock interview sessions, soft skill training and sessions with industry mentors.
Will I get help building my resume?
Yes, we conduct dedicated resume building workshops before the placement assistance phase begins.
Will there be mock interviews before real interviews?
Yes, our mentors conduct multiple mock interview rounds to prepare you for real interviews.
Is there any community support available during the bootcamp?
Yes, we have an active student community where you can discuss doubts, share resources and collaborate on projects.
"""

## 3. Preprocessing: Tokenization & Vocabulary

First we download NLTK's tokenizer, then break the whole document into individual word tokens (all lowercase), and finally build a vocabulary that maps every unique word to a number.

In [3]:
# Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Error loading punkt: <urlopen error pathsec.urlopen: no
[nltk_data]     validated address for host
[nltk_data]     'raw.githubusercontent.com'; refusing to connect by
[nltk_data]     unvalidated hostname>
[nltk_data] Error loading punkt_tab: <urlopen error pathsec.urlopen:
[nltk_data]     no validated address for host
[nltk_data]     'raw.githubusercontent.com'; refusing to connect by
[nltk_data]     unvalidated hostname>


False

In [4]:
# tokenize the whole document
tokens = word_tokenize(document.lower())
len(tokens)

1046

In [5]:
# build vocab: word -> unique index
vocab = {'<unk>': 0}

for token in Counter(tokens).keys():
    if token not in vocab:
        vocab[token] = len(vocab)

len(vocab)

333

## 4. Build Training Sequences

We turn every line of the document into progressively growing sequences.
For example the sentence `"the course fee is"` becomes:
- `[the, course] -> fee`
- `[the, course, fee] -> is`

This teaches the model to predict the next word given all the previous words.

In [6]:
def text_to_indices(sentence, vocab):
    numerical_sentence = []
    for token in sentence:
        if token in vocab:
            numerical_sentence.append(vocab[token])
        else:
            numerical_sentence.append(vocab['<unk>'])
    return numerical_sentence

In [7]:
input_sentences = document.split('\n')

input_numerical_sentences = []
for sentence in input_sentences:
    input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))

len(input_numerical_sentences)

85

In [8]:
training_sequence = []
for sentence in input_numerical_sentences:
    for i in range(1, len(sentence)):
        training_sequence.append(sentence[:i+1])

len(training_sequence)

962

## 5. Padding

All training sequences must be the same length to be fed into the model as a batch. We pad the shorter ones with `0` at the beginning.

In [9]:
len_list = [len(sequence) for sequence in training_sequence]
max(len_list)

33

In [10]:
padded_training_sequence = []
for sequence in training_sequence:
    padded_training_sequence.append([0] * (max(len_list) - len(sequence)) + sequence)

padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)
padded_training_sequence.shape

torch.Size([962, 33])

In [11]:
# X = input words, y = the word to predict (the last one)
X = padded_training_sequence[:, :-1]
y = padded_training_sequence[:, -1]

X.shape, y.shape

(torch.Size([962, 32]), torch.Size([962]))

## 6. PyTorch Dataset & DataLoader

In [12]:
class CustomDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [13]:
dataset = CustomDataset(X, y)
len(dataset)

962

In [14]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

## 7. Define the LSTM Model

A small model:
- **Embedding** layer turns word indices into dense vectors
- **LSTM** layer reads the sequence and remembers context
- **Linear** layer converts the LSTM output into a score for every word in the vocabulary

In [15]:
class LSTMModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 100)
        self.lstm = nn.LSTM(100, 150, batch_first=True)
        self.fc = nn.Linear(150, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
        output = self.fc(final_hidden_state.squeeze(0))
        return output

In [16]:
model = LSTMModel(len(vocab))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

LSTMModel(
  (embedding): Embedding(333, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=333, bias=True)
)

## 8. Train the Model

In [17]:
epochs = 35
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [18]:
# training loop
for epoch in range(epochs):
    total_loss = 0

    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()

        total_loss = total_loss + loss.item()

    print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 175.9922
Epoch: 2, Loss: 157.9176
Epoch: 3, Loss: 141.5570
Epoch: 4, Loss: 132.8388
Epoch: 5, Loss: 120.4441
Epoch: 6, Loss: 106.8179
Epoch: 7, Loss: 96.9882
Epoch: 8, Loss: 87.4092
Epoch: 9, Loss: 78.2814
Epoch: 10, Loss: 68.2488
Epoch: 11, Loss: 60.3791
Epoch: 12, Loss: 54.2109
Epoch: 13, Loss: 48.6809
Epoch: 14, Loss: 41.6145
Epoch: 15, Loss: 37.9752
Epoch: 16, Loss: 35.3873
Epoch: 17, Loss: 30.3575
Epoch: 18, Loss: 27.2401
Epoch: 19, Loss: 23.5716
Epoch: 20, Loss: 21.0168
Epoch: 21, Loss: 19.1280
Epoch: 22, Loss: 17.7382
Epoch: 23, Loss: 17.0135
Epoch: 24, Loss: 14.7923
Epoch: 25, Loss: 15.2467
Epoch: 26, Loss: 13.0197
Epoch: 27, Loss: 13.4183
Epoch: 28, Loss: 11.8348
Epoch: 29, Loss: 11.0907
Epoch: 30, Loss: 10.7441
Epoch: 31, Loss: 10.0404
Epoch: 32, Loss: 9.3200
Epoch: 33, Loss: 9.6732
Epoch: 34, Loss: 9.2179
Epoch: 35, Loss: 8.4104


## 9. Check Accuracy

In [19]:
def calculate_accuracy(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            _, predicted = torch.max(outputs, dim=1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = correct / total * 100
    return accuracy

eval_dataloader = DataLoader(dataset, batch_size=32, shuffle=False)
accuracy = calculate_accuracy(model, eval_dataloader, device)
print(f"Model Accuracy: {accuracy:.2f}%")

Model Accuracy: 93.87%


## 10. Generate Predictions

Given some starting text, predict the next word (and repeat to generate several words in a row).

In [20]:
def prediction(model, vocab, text):
    # tokenize
    tokenized_text = word_tokenize(text.lower())

    # text -> numerical indices
    numerical_text = text_to_indices(tokenized_text, vocab)

    # padding
    padded_text = torch.tensor([0] * (max(len_list) - 1 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)

    # send to model
    output = model(padded_text.to(device))

    # predicted index
    value, index = torch.max(output, dim=1)

    # merge with text
    return text + " " + list(vocab.keys())[index]

In [21]:
prediction(model, vocab, "the course fee is")

'the course fee is rs'

In [42]:
import time
num_tokens = 15
input_text = "The course is"

for i in range(num_tokens):
    output_text = prediction(model, vocab, input_text)
    print(output_text)
    input_text = output_text
    time.sleep(0.2)

The course is rs
The course is rs 999
The course is rs 999 per
The course is rs 999 per month
The course is rs 999 per month ,
The course is rs 999 per month , and
The course is rs 999 per month , and you
The course is rs 999 per month , and you can
The course is rs 999 per month , and you can pay
The course is rs 999 per month , and you can pay it
The course is rs 999 per month , and you can pay it as
The course is rs 999 per month , and you can pay it as a
The course is rs 999 per month , and you can pay it as a monthly
The course is rs 999 per month , and you can pay it as a monthly subscription
The course is rs 999 per month , and you can pay it as a monthly subscription .


## 11. Save the Model

We save the trained weights + vocabulary so the FastAPI backend can load them later to make real predictions (see the `backend/` folder).

In [23]:
import json

torch.save(model.state_dict(), "model.pth")

with open("vocab.json", "w") as f:
    json.dump(vocab, f)

print("Saved model.pth and vocab.json")

Saved model.pth and vocab.json
